# Refined Shape Probing — existing 30 frozen checkpoints

This is the **replacement measurement stage** for the original mean-pooled Shape probe. It uses the same ten trained models and their three checkpoints, **without retraining the ViT**. Run all five seeds immediately; there is no pilot phase.

The comparison is a fixed 2×2 design: `mean` versus spatially ordered `5×5` patch features, each with `original` versus recolored/color-balanced diagnostic training scenes. The random-encoder and raw-image silhouette baselines are controls. The original probing outputs are kept unchanged.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'outputs/medium_world/dataset'
OUT_DIR = PROJECT_ROOT / 'outputs/medium_world/experiments/representation_diagnostics/five_seeds'
SEEDS = (42, 43, 44, 45, 46)
SCHEDULES = ('prereq_mixed_pair_focus', 'mixed')
STEPS = (5400, 7200, 9000)
SPLITS = ('id_dev', 'ood_dev')
N_TRAIN_SCENES = 1200
N_EVAL_SCENES = {'id_dev': 512, 'ood_dev': 400}
DEVICE = 'cuda'

In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.medium_visual_world.learning import MediumDataset, AttributeBindingViT, set_seed
from src.medium_visual_world.representation_probing import (
    distinct_scenes, extract_spatial_object_features, extract_silhouette_features,
    recolor_probe_scenes, shape_color_weights, fit_spatial_shape_decoder,
    evaluate_shape_decoder,
)

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable. Set DEVICE to cpu if appropriate')
device = torch.device(DEVICE)
train = MediumDataset(DATA_DIR / 'train.jsonl')
dev = {split: MediumDataset(DATA_DIR / f'{split}.jsonl') for split in SPLITS}
manifest = pd.read_csv(OUT_DIR / 'checkpoint_manifest.csv')
train_rows = distinct_scenes(train.rows, N_TRAIN_SCENES, 313)
recolored_rows = recolor_probe_scenes(train_rows, 20261110)
eval_rows = {
    split: distinct_scenes(dataset.rows, N_EVAL_SCENES[split], 314 + i)
    for i, (split, dataset) in enumerate(dev.items())
}

original_meta, original_silhouette = extract_silhouette_features(train_rows)
recolored_meta, recolored_silhouette = extract_silhouette_features(recolored_rows)
recolor_weights, balanced_cells = shape_color_weights(recolored_meta)
original_cells = (original_meta.groupby(['shape', 'color']).size()
                  .rename('n_objects').reset_index())
original_cells['regime'] = 'original'
balanced_cells['regime'] = 'recolored_balanced'
balance_table = pd.concat([original_cells, balanced_cells], ignore_index=True)
balance_table.to_csv(OUT_DIR / 'refined_probe_training_balance.csv', index=False)
IMAGE_CACHE = {}  # Reuse identical rendered scenes across 30 frozen checkpoints
print('Train scenes:', len(train_rows), '| Dev scenes:',
      {name: len(rows) for name, rows in eval_rows.items()})
display(balance_table.pivot_table(index='shape', columns=['regime', 'color'],
                                  values='n_objects', fill_value=0))

Train scenes: 1200 | Dev scenes: {'id_dev': 512, 'ood_dev': 400}


regime   original                             recolored_balanced         \
color        blue  green purple    red yellow               blue  green   
shape                                                                     
circle      293.0    0.0  348.0  300.0  302.0              243.0  258.0   
crescent    273.0  294.0  308.0    0.0  276.0              214.0  248.0   
square        0.0  325.0  311.0  298.0  319.0              258.0  232.0   
triangle    278.0  310.0    0.0  302.0  289.0              225.0  228.0   

regime                         
color    purple    red yellow  
shape                          
circle    238.0  254.0  250.0  
crescent  229.0  202.0  258.0  
square    252.0  249.0  262.0  
triangle  239.0  248.0  239.0

## Sanity check: can a model-independent pixel silhouette reveal Shape?

This is a diagnostic control, not a replacement architecture. A fixed crop centered on each object contains only pixels and a foreground/background threshold, without Shape-derived masks. The same classifier and evaluation splits are used.

In [3]:
silhouette_train = {}
for regime, rows in [('original', train_rows),
                     ('recolored_balanced', recolored_rows)]:
    meta = original_meta if regime == 'original' else recolored_meta
    features = (original_silhouette if regime == 'original'
                else recolored_silhouette)
    weights = recolor_weights if regime == 'recolored_balanced' else None
    probe = fit_spatial_shape_decoder(features, meta['shape'].to_numpy(), weights)
    silhouette_train[regime] = probe

silhouette_metrics = []
silhouette_predictions = []
for split, rows in eval_rows.items():
    meta, features = extract_silhouette_features(rows)
    truth = meta['shape'].to_numpy()
    for regime, probe in silhouette_train.items():
        result = evaluate_shape_decoder(probe, features, truth)
        silhouette_metrics.append({
            'regime': regime, 'split': split,
            'accuracy': result['accuracy'], 'macro_f1': result['macro_f1'],
            'n_eval_objects': len(meta)
        })
        out = meta.copy()
        out['split'] = split
        out['regime'] = regime
        out['target'] = truth
        out['prediction'] = result['predictions']
        out['correct'] = truth == result['predictions']
        silhouette_predictions.append(out)
pd.DataFrame(silhouette_metrics).to_csv(
    OUT_DIR / 'refined_probe_silhouette_metrics.csv', index=False)
pd.concat(silhouette_predictions, ignore_index=True).to_csv(
    OUT_DIR / 'refined_probe_silhouette_predictions.csv', index=False)
display(pd.DataFrame(silhouette_metrics))

,regime,split,accuracy,macro_f1,n_eval_objects
0,original,id_dev,1.0,1.0,2044
1,recolored_balanced,id_dev,1.0,1.0,2044
2,original,ood_dev,1.0,1.0,1677
3,recolored_balanced,ood_dev,1.0,1.0,1677


## Frozen encoder readouts

For each checkpoint, train a linear Shape decoder **once on training scenes**, then evaluate the same decoder on ID-dev and OOD-dev. A spatial probe sees 25 ordered feature vectors; the matched mean probe sees their average. Recolored training scenes use cyclic shifts of the palette without changing Shape or geometry; inverse Shape×Color frequencies equalize training weight across all 20 combinations.

OOD-dev remains held out for **ViT training**, but the recolored probe has seen all combinations during its own fitting: report that comparison as *probe accessibility under color-balanced training*, **not** as unseen-combination generalization of the probe.

In [4]:
def load_checkpoint(schedule, seed, step):
    record = manifest.loc[
        (manifest['schedule'] == schedule)
        & (manifest['seed'] == seed)
        & (manifest['step'] == step)
    ].iloc[0]
    with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
        saved = torch.load(Path(record['path']), map_location='cpu', weights_only=True)
    model = AttributeBindingViT(**saved['model_config']).to(device)
    model.load_state_dict(saved['model'], strict=True)
    return model.eval(), record['checkpoint_sha256']


def random_encoder(seed):
    set_seed(seed + 19293)
    return AttributeBindingViT(dim=48, patch_size=4, depth=1).to(device).eval()


def run_decoder_grid(model, baseline, seed):
    metadata = {}
    features = {}
    for regime, rows in [('original', train_rows),
                         ('recolored_balanced', recolored_rows)]:
        metadata[regime], features[regime] = extract_spatial_object_features(
            model, rows, device, image_cache=IMAGE_CACHE)
    evaluated = {
        split: extract_spatial_object_features(model, rows, device, image_cache=IMAGE_CACHE)
        for split, rows in eval_rows.items()
    }
    metrics, predictions = [], []
    for stage in ('patch', 'encoder', 'fused'):
        for regime in ('original', 'recolored_balanced'):
            source = metadata[regime]
            y_train = source['shape'].to_numpy()
            weights = recolor_weights if regime == 'recolored_balanced' else None
            for readout in ('mean', 'spatial'):
                x_train = features[regime][stage][readout]
                classifier = fit_spatial_shape_decoder(x_train, y_train, weights)
                for split, (eval_meta, eval_features) in evaluated.items():
                    truth = eval_meta['shape'].to_numpy()
                    x_eval = eval_features[stage][readout]
                    result = evaluate_shape_decoder(classifier, x_eval, truth)
                    metrics.append({
                        'baseline': baseline, 'stage': stage, 'regime': regime,
                        'readout': readout, 'split': split,
                        'accuracy': result['accuracy'], 'macro_f1': result['macro_f1'],
                        'n_train_objects': len(source),
                        'n_eval_objects': len(eval_meta),
                    })
                    if stage == 'fused':
                        output = eval_meta.copy()
                        output['baseline'] = baseline
                        output['stage'] = stage
                        output['regime'] = regime
                        output['readout'] = readout
                        output['split'] = split
                        output['target'] = truth
                        output['prediction'] = result['predictions']
                        output['correct'] = truth == result['predictions']
                        predictions.append(output)
    if baseline == 'trained':
        meta = metadata['recolored_balanced']
        x_train = features['recolored_balanced']['fused']['spatial']
        shuffled = np.random.default_rng(seed + 713).permutation(
            meta['shape'].to_numpy())
        control = fit_spatial_shape_decoder(x_train, shuffled, recolor_weights)
        for split, (eval_meta, eval_features) in evaluated.items():
            result = evaluate_shape_decoder(
                control, eval_features['fused']['spatial'],
                eval_meta['shape'].to_numpy())
            metrics.append({
                'baseline': 'shuffled_label_control', 'stage': 'fused',
                'regime': 'recolored_balanced', 'readout': 'spatial',
                'split': split, 'accuracy': result['accuracy'],
                'macro_f1': result['macro_f1'],
                'n_train_objects': len(meta),
                'n_eval_objects': len(eval_meta),
            })
    return pd.DataFrame(metrics), pd.concat(predictions, ignore_index=True)

In [5]:
random_results = {}
for schedule in SCHEDULES:
    for seed in SEEDS:
        for step in STEPS:
            trained, ckhash = load_checkpoint(schedule, seed, step)
            trained_metrics, trained_predictions = run_decoder_grid(trained, 'trained', seed)
            del trained
            gc.collect()
            if seed not in random_results:
                control = random_encoder(seed)
                random_results[seed] = run_decoder_grid(control, 'random', seed)
                del control
                gc.collect()
            random_metrics, random_predictions = random_results[seed]
            metrics = pd.concat([trained_metrics, random_metrics], ignore_index=True)
            predictions = pd.concat([trained_predictions, random_predictions], ignore_index=True)
            for frame in (metrics, predictions):
                frame['schedule'] = schedule
                frame['seed'] = seed
                frame['step'] = step
                frame['checkpoint_sha256'] = ckhash
            suffix = f'{schedule}_seed{seed}_step{step}'
            metrics.to_csv(OUT_DIR / f'refined_probe_metrics_{suffix}.csv', index=False)
            predictions.to_csv(OUT_DIR / f'refined_probe_predictions_{suffix}.csv', index=False)
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            print('Completed refined probes:', suffix, flush=True)

Completed refined probes: prereq_mixed_pair_focus_seed42_step5400
Completed refined probes: prereq_mixed_pair_focus_seed42_step7200
Completed refined probes: prereq_mixed_pair_focus_seed42_step9000
Completed refined probes: prereq_mixed_pair_focus_seed43_step5400
Completed refined probes: prereq_mixed_pair_focus_seed43_step7200
Completed refined probes: prereq_mixed_pair_focus_seed43_step9000
Completed refined probes: prereq_mixed_pair_focus_seed44_step5400
Completed refined probes: prereq_mixed_pair_focus_seed44_step7200
Completed refined probes: prereq_mixed_pair_focus_seed44_step9000
Completed refined probes: prereq_mixed_pair_focus_seed45_step5400
Completed refined probes: prereq_mixed_pair_focus_seed45_step7200
Completed refined probes: prereq_mixed_pair_focus_seed45_step9000
Completed refined probes: prereq_mixed_pair_focus_seed46_step5400
Completed refined probes: prereq_mixed_pair_focus_seed46_step7200
Completed refined probes: prereq_mixed_pair_focus_seed46_step9000
Completed 

## Analyze next

Run `representation_probing_analysis.ipynb` and then `representation_evidence_analysis.ipynb`. Both include the new refined-probe comparisons. All earlier diagnostic outputs are preserved; this notebook does not run any ViT optimization.